# Biohub: compare checkpoints on the same ROI

Copy these code cells into the **end of the existing Kaggle pilot notebook** and run only the new cells in the same live session. This reuses the 2,000-iteration candidates and calibrated graph. It runs tiled inference for checkpoints 500, 1000, and 1500, then applies the **same fixed ILP costs** to each. It does not train again or alter original outputs. Hidden-GT columns are exploratory diagnostics; choose the next training duration from the visible-candidate trend, not the hidden-GT maximum.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
import torch
from IPython.display import display

DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
ROOT = Path('/kaggle/working/biohub_random_hide')
REPO = Path('/kaggle/working/linajea')
ITERATIONS = (500, 1000, 1500, 2000)
PINNED_COMPARISON_COMMIT = None  # Optional SHA after pushing this code

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU for checkpoint inference')
required = [ROOT / 'split.json', ROOT / 'visible.geff',
            ROOT / 'prediction' / 'candidate_nodes.csv',
            ROOT / 'prediction' / 'candidate_edges.csv',
            ROOT / 'prediction_calibrated' / 'calibration.json',
            ROOT / 'prediction_calibrated' / 'pred_nodes.csv',
            ROOT / 'prediction_calibrated' / 'pred_edges.csv']
required += [ROOT / 'train' / f'train_net_checkpoint_{it}' for it in ITERATIONS]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f'Missing existing pilot output: {path}')

subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'biohub-io'], check=True)
commit = PINNED_COMPARISON_COMMIT or subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'origin/biohub-io'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', commit], check=True)
commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
comparison_dir = ROOT / 'checkpoint_comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
(comparison_dir / 'code_commit.txt').write_text(commit + '\n')
print('Code commit:', commit)
print('Checkpoints:', ITERATIONS)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
command = [sys.executable, '-m', 'run_scripts.compare_biohub_checkpoints',
           '--data-dir', str(DATA_DIR), '--output-dir', str(ROOT),
           '--iterations', *map(str, ITERATIONS),
           '--reuse-iteration', '2000', '--device', 'cuda']
with (comparison_dir / 'run.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line)
        log.flush()
        if ('Iteration ' in line or 'Traceback' in line or 'Error' in line):
            print(line.rstrip())
    if process.wait() != 0:
        raise RuntimeError(f'Checkpoint comparison failed. See {comparison_dir / "run.log"}')
print('Comparison saved to:', comparison_dir)

In [ ]:
# Final comparison table at the fixed 4 µm match distance.
summary = pd.read_csv(comparison_dir / 'summary_4um.csv')
all_metrics = pd.read_csv(comparison_dir / 'metrics_all_thresholds.csv')
visible_total = int(all_metrics['visible_total'].iloc[0])
hidden_total = int(all_metrics['hidden_total'].iloc[0])
hidden_edge_total = int(all_metrics['hidden_edges_total'].iloc[0])

def fraction(value, total):
    return f'{int(value)}/{total} ({value / total:.0%})' if total else 'n/a'

table = pd.DataFrame({
    'Iteration': summary['iteration'],
    'Candidate nodes': summary['candidate_nodes'],
    'Visible GT matched (candidate)': [
        fraction(v, visible_total) for v in summary['candidate_visible_recovered_4um']],
    'Hidden GT matched (candidate)': [
        fraction(v, hidden_total) for v in summary['candidate_hidden_recovered_4um']],
    'Selected nodes': summary['selected_nodes'],
    'Hidden GT matched (after ILP)': [
        fraction(v, hidden_total) for v in summary['selected_hidden_recovered_4um']],
    'Hidden edges (candidate)': [
        fraction(v, hidden_edge_total) for v in summary['candidate_hidden_edges_4um']],
    'Hidden edges (after ILP)': [
        fraction(v, hidden_edge_total) for v in summary['selected_hidden_edges_4um']],
    'Mean candidate score': summary['candidate_score_mean'].round(3),
})
display(table.set_index('Iteration'))
report = json.loads((comparison_dir / 'comparison.json').read_text())
print('Visible-only diagnostic checkpoint:',
      report['suggested_iteration_from_visible_only'])
print('Full 2/4/7 µm metrics:', comparison_dir / 'metrics_all_thresholds.csv')
print('This is a same-sequence exploratory comparison, not independent validation.')